In [1]:
import shutil
import os

source = '/kaggle/input/notebooks/hasnatabdullah/dataset-creation/bnmc_hf_dataset'
destination = '/kaggle/working/bnmc_hf_dataset'

if not os.path.exists(destination):
    shutil.copytree(source, destination)
    print("copied")

copied


In [2]:
from datasets import load_from_disk

ds = load_from_disk('/kaggle/working/bnmc_hf_dataset')


In [3]:
from sklearn.preprocessing import MultiLabelBinarizer

def split_labels(example):
    return {'label_list': [l.strip() for l in example['labels'].split(',')]}

ds = ds.map(split_labels)

mlb = MultiLabelBinarizer()
mlb.fit(ds['train']['label_list'])

mlb.classes_

Map:   0%|          | 0/36144 [00:00<?, ? examples/s]

Map:   0%|          | 0/4519 [00:00<?, ? examples/s]

Map:   0%|          | 0/4518 [00:00<?, ? examples/s]

array(['অন্যান্য-খেলা', 'অপরাধ', 'অপহরণ', 'অর্থনীতি', 'আইন',
       'আইন-শৃঙ্খলা', 'আওমীলীগ', 'আন্তর্জাতিক', 'আবহাওয়া', 'আবাসন',
       'উৎসব', 'কূটনীতি', 'কৃষি', 'ক্রিকেট', 'ক্রীড়া', 'জামায়াত',
       'জীবনধারা', 'জেলা সংবাদ', 'ঢাকা', 'দুর্ঘটনা', 'দুর্নীতি',
       'দুর্যোগ', 'ধর্ম', 'নির্বাচন', 'পরিবেশ', 'পর্যটন', 'প্রযুক্তি',
       'ফুটবল', 'বাণিজ্য', 'বি এন পি', 'বিজ্ঞান', 'ব্যাংকিং',
       'যানজট/ট্রাফিক', 'রাজনীতি', 'রোগ-বিস্তার', 'শিক্ষা',
       'শিল্প-কারখানা', 'শেয়ারবাজার', 'সংসদ', 'সংস্কৃতি', 'সমাজ',
       'সরকারি-নীতি', 'সেলিব্রিটি', 'স্বাস্থ্য', 'হত্যা'], dtype=object)

In [4]:
len(mlb.classes_)

45

In [5]:
def binarize(batch):
    return {'binarized_labels': mlb.transform(batch['label_list'])}

dataset_binarized = ds.map(binarize, batched = True)
dataset_binarized = dataset_binarized.remove_columns(["labels", "label_list"])
dataset_binarized['test']['binarized_labels'][0]

Map:   0%|          | 0/36144 [00:00<?, ? examples/s]

Map:   0%|          | 0/4519 [00:00<?, ? examples/s]

Map:   0%|          | 0/4518 [00:00<?, ? examples/s]

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [6]:
from transformers import AutoTokenizer

model_name = "sagorsarker/bangla-bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [7]:


def tokenizer_fun(batch):
    tokenized= tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length', 
        max_length=256
    )
    tokenized['labels'] = [list(map(float, l)) for l in batch['binarized_labels']]
    return tokenized

tokenized_dataset = dataset_binarized.map(
        tokenizer_fun, batched = True, remove_columns = dataset_binarized['train'].column_names
    )

The OrderedVocab you are attempting to save contains holes for indices [1015, 1016, 1017, 1018, 1053, 1054, 1055, 1056, 1057, 1060, 1061, 1062, 1064, 1065, 1066, 1067, 1068, 1069, 1070, 1071, 1072, 1073, 1074, 1075, 1076, 1077, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1087, 1088, 1089, 1090, 1091, 1092, 1093, 1094, 1095, 1099, 1101, 1112, 1113, 1556, 1557, 1568], your vocabulary could be corrupted !


Map:   0%|          | 0/36144 [00:00<?, ? examples/s]

The OrderedVocab you are attempting to save contains holes for indices [1015, 1016, 1017, 1018, 1053, 1054, 1055, 1056, 1057, 1060, 1061, 1062, 1064, 1065, 1066, 1067, 1068, 1069, 1070, 1071, 1072, 1073, 1074, 1075, 1076, 1077, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1087, 1088, 1089, 1090, 1091, 1092, 1093, 1094, 1095, 1099, 1101, 1112, 1113, 1556, 1557, 1568], your vocabulary could be corrupted !


Map:   0%|          | 0/4519 [00:00<?, ? examples/s]

The OrderedVocab you are attempting to save contains holes for indices [1015, 1016, 1017, 1018, 1053, 1054, 1055, 1056, 1057, 1060, 1061, 1062, 1064, 1065, 1066, 1067, 1068, 1069, 1070, 1071, 1072, 1073, 1074, 1075, 1076, 1077, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1087, 1088, 1089, 1090, 1091, 1092, 1093, 1094, 1095, 1099, 1101, 1112, 1113, 1556, 1557, 1568], your vocabulary could be corrupted !


Map:   0%|          | 0/4518 [00:00<?, ? examples/s]

In [8]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 36144
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 4519
    })
    val: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 4518
    })
})

In [9]:
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    get_linear_schedule_with_warmup,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score, accuracy_score
import pickle


2026-03-08 07:22:21.597512: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772954541.979473      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772954542.089939      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=45,
    problem_type="multi_label_classification",  
)

model.safetensors:   0%|          | 0.00/660M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sagorsarker/bangla-bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
# ==================== Metrics ====================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits))
    preds = (probs > 0.5).int().numpy()
    labels = labels.astype(int)

    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    accuracy = accuracy_score(labels, preds)

    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "accuracy": accuracy,
    }

In [12]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("wb_sec")


In [13]:
!wandb login --relogin $secret_value_0


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [14]:
training_args = TrainingArguments(
    output_dir="bert_exp2",
    run_name = 'bert_exp2',
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=0,  
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    greater_is_better=True,
    report_to="wandb",  
    fp16=torch.cuda.is_available(),  
)

In [15]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["val"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [16]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: hasnatabdullah47 (uolo) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.21.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260308_072304-o3c27bcj
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bert_exp2
wandb: ⭐️ View project at https://wandb.ai/uolo/huggingface
wandb: 🚀 View run at https://wandb.ai/uolo/huggingface/runs/o3c27bcj
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Accuracy
1,0.132500,0.118425,0.732978,0.400120,0.265383
2,0.102200,0.099958,0.776904,0.507885,0.313192
3,0.086800,0.093220,0.795957,0.585096,0.333555
4,0.074200,0.090642,0.796475,0.589919,0.332005
5,0.065100,0.090223,0.799119,0.609558,0.343958
6,0.056900,0.090024,0.803371,0.636291,0.343072
7,0.049600,0.091819,0.802139,0.644229,0.337760
8,0.043100,0.094410,0.802225,0.655012,0.339973
9,0.038400,0.096862,0.799976,0.655978,0.335104


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked t

TrainOutput(global_step=10170, training_loss=0.075706530086176, metrics={'train_runtime': 9739.1987, 'train_samples_per_second': 55.668, 'train_steps_per_second': 1.74, 'total_flos': 4.281100901270323e+16, 'train_loss': 0.075706530086176, 'epoch': 9.0})

In [17]:
test_results = trainer.evaluate(tokenized_dataset["test"])
print(test_results)


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'eval_loss': 0.08503211289644241, 'eval_f1_micro': 0.8139373662477017, 'eval_f1_macro': 0.6311229386316399, 'eval_accuracy': 0.36069926974994465, 'eval_runtime': 43.2007, 'eval_samples_per_second': 104.605, 'eval_steps_per_second': 3.287, 'epoch': 9.0}


In [18]:
trainer.save_model("final_bangla_multilabel_model")
tokenizer.save_pretrained("final_bangla_multilabel_model")

The OrderedVocab you are attempting to save contains holes for indices [1015, 1016, 1017, 1018, 1053, 1054, 1055, 1056, 1057, 1060, 1061, 1062, 1064, 1065, 1066, 1067, 1068, 1069, 1070, 1071, 1072, 1073, 1074, 1075, 1076, 1077, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1087, 1088, 1089, 1090, 1091, 1092, 1093, 1094, 1095, 1099, 1101, 1112, 1113, 1556, 1557, 1568], your vocabulary could be corrupted !


('final_bangla_multilabel_model/tokenizer_config.json',
 'final_bangla_multilabel_model/special_tokens_map.json',
 'final_bangla_multilabel_model/vocab.txt',
 'final_bangla_multilabel_model/added_tokens.json',
 'final_bangla_multilabel_model/tokenizer.json')